# Scanpy Pipeline

## Source: https://scanpy.readthedocs.io/en/stable/tutorials/index.html

| Package           | Version |
| ----------------- | ------- |
| Python            |  3.12.3 
| scanpy            | 1.12    |
| pandas            | 2.3.3   |
| numpy             | 2.4.4   |
| matplotlib        | 3.10.8  |
| matplotlib-inline | 0.2.1   |
| harmonypy         | 0.2.0   |
| anndata           | 0.12.10 |

In [ ]:
# 1. Load necessary libraries
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import matplotlib_inline.backend_inline
import harmonypy as hm


# Font Style and settings
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelweight'] = 'bold'
os.makedirs(sc.settings.figdir, exist_ok=True)

In [ ]:
## 1.1 Set reproducibility seeds
# Set seeds for reproducibility
random.seed(0)
np.random.seed(0)

In [ ]:
## 1.2 Set global parameters
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)

# Set figures setthings
sc.set_figure_params(
    scanpy=True, 
    dpi=300,
    dpi_save=300, 
    frameon=False, 
    figsize=(4, 4), 
    facecolor='white'
)
sc.settings.figdir = "./figures"

matplotlib_inline.backend_inline.set_matplotlib_formats('png', 'pdf')  

# Create output directories if they don't exist
os.makedirs('adata_object', exist_ok=True)

In [ ]:
# 2. Load Cellbender-formatted data 
control1 = sc.read_10x_h5('../Cellbender/cellbender_SRR28227295_output_file_filtered.h5', gex_only=True)
control2 = sc.read_10x_h5('../Cellbender/cellbender_SRR28227293_output_file_filtered.h5', gex_only=True)

In [ ]:
## 2.1 Add batch effects identifiers to anndata objects

control1.obs["sample"] = "control1"
control2.obs["sample"] = "control2"

In [ ]:
# Check how many cells and genes pre-QC
print(f"Remaining cells in control 1: {control1.n_obs}, Remaining genes in control 1: {control1.n_vars}")
print(f"Remaining cells in control 2: {control2.n_obs}, Remaining genes in control 2: {control2.n_vars}")

In [ ]:
# 3. Quality Control (QC)
## 3.1 Manual mito, ribo and hibo genes annotation

# A manual list of mitochondrial, ribosomal, and hemoglobin genes was compiled to carry out quality control.
# List of known mitochondrial; hemoglobin and ribosomal genes for rainbow trout
mito_genes = pd.read_csv("manual_gene_list/mito.tsv", sep="\t")
hemoglobin = pd.read_csv("manual_gene_list/hemoglobin_genes.tsv", sep="\t")
rpl = pd.read_csv("manual_gene_list/rpl_ribo.tsv", sep="\t")
rps = pd.read_csv("manual_gene_list/rps_ribo.tsv", sep="\t")

mito_genes = mito_genes['Symbol'].values
hb = hemoglobin['Symbol'].values
ribo = np.concatenate((rpl['Symbol'].values, rps['Symbol'].values))

In [ ]:
# Identification of mitochondrial genes in each dataset

control1.var["mt"] = control1.var_names.isin(mito_genes)
control2.var["mt"] = control2.var_names.isin(mito_genes)

# Identification of ribosomal genes in each dataset

control1.var["ribo"] = control1.var_names.isin(ribo)
control2.var["ribo"] = control2.var_names.isin(ribo)

# Identification of hemoglobin genes in each dataset

control1.var["hb"] = control1.var_names.isin(hb)
control2.var["hb"] = control2.var_names.isin(hb)

In [ ]:
## 3.2 Calculate QC metrics

sc.pp.calculate_qc_metrics(control1, expr_type='counts', var_type='genes', qc_vars=["mt", "ribo", "hb"],
                           percent_top=(50, 100, 200, 500), layer=None, use_raw=False, 
                           inplace=True, log1p=True, parallel=None)

sc.pp.calculate_qc_metrics(control2, expr_type='counts', var_type='genes', qc_vars=["mt", "ribo", "hb"], 
                           percent_top=(50, 100, 200, 500), layer=None, use_raw=False, 
                           inplace=True, log1p=True, parallel=None)

In [ ]:
## 3.3 Plot Pre-QC

sc.pl.violin(
    control1,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"],
    jitter=0.4,
    multi_panel=True,
    save='_pre_QC_plot_control1.png'
)


sc.pl.violin(
    control2,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"],
    jitter=0.4,
    multi_panel=True,
    save='_pre_QC_plot_control2.png'
)

In [ ]:
# Per-sample filtering

sc.pp.filter_cells(control1, min_genes=200)
sc.pp.filter_cells(control2, min_genes=200)

sc.pp.filter_genes(control1, min_cells=3)
sc.pp.filter_genes(control2, min_cells=3)

In [ ]:
# Sample and experiment specific filtering

# Filter out cells with detected gene number > 3,500 

control1 = control1[control1.obs['n_genes_by_counts'] <= 3500].copy()
control2 = control2[control2.obs['n_genes_by_counts'] <= 3500].copy()

# Filter out cells with total counts > 15,000
control1 = control1[control1.obs['total_counts'] <= 15000].copy()
control2 = control2[control2.obs['total_counts'] <= 15000].copy()

In [ ]:
# Filter out cells with >25% mitochondrial genes

control1 = control1[control1.obs['pct_counts_mt'] <= 25].copy()
control2 = control2[control2.obs['pct_counts_mt'] <= 25].copy()

In [ ]:
# Filter out cells with >1% hemoglobin genes

control1 = control1[control1.obs['pct_counts_hb'] <= 1].copy()
control2 = control2[control2.obs['pct_counts_hb'] <=1].copy()

In [ ]:
## 3.5 Sanity check how many cells and genes remain after QC filtering

print(f"Remaining cells in control 1: {control1.n_obs}, Remaining genes in control 1: {control1.n_vars}")
print(f"Remaining cells in control 2: {control2.n_obs}, Remaining genes in control 2: {control2.n_vars}")

In [ ]:
## 3.3 Plot Post-QC

sc.pl.violin(
    control1,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"],
    jitter=0.4,
    multi_panel=True,
    save='_post_QC_plot_control1.png'
)


sc.pl.violin(
    control2,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"],
    jitter=0.4,
    multi_panel=True,
    save='_post_QC_plot_control2.png'
)

In [ ]:
# Merge

adata = control1.concatenate(control2, batch_key="sample", batch_categories=["control1", "control2"])

In [ ]:
# Save raw counts
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Doublet detection

sc.pp.scrublet(adata, batch_key='sample')

In [ ]:
# Remove doublets before downstream processing
adata = adata[~adata.obs["predicted_doublet"].astype(bool)].copy()

In [ ]:
# Normalise & log transform
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
# Logarithmize the data
sc.pp.log1p(adata)

In [ ]:
# Freeze the normlaized log counts
adata.raw = adata

In [ ]:
# HVG, PCA and Neighbours

sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="sample")

In [ ]:
sc.pp.scale(adata, max_value=10)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
# Harmony Integration
# https://github.com/slowkow/harmonypy

hmpy = hm.run_harmony(adata.obsm["X_pca"], adata.obs, "sample")

In [ ]:
# hmpy.Z_corr already in the right orientation, so no need to transpose
adata.obsm["X_pca_harmony"] = hmpy.Z_corr

In [ ]:
print(adata.obsm["X_pca_harmony"].shape)
print(adata.obsm["X_pca"].shape)

In [ ]:
sc.pp.neighbors(adata, use_rep="X_pca_harmony")

In [ ]:
sc.tl.umap(adata)

In [ ]:
# umap check
sc.pl.umap(adata, color="sample", save= "_check.png")

In [ ]:
for res in [0.3, 0.5, 0.6, 0.7, 0.8, 1.0, 1.5]:
    sc.tl.leiden(adata, resolution=res,
                 flavor="igraph", n_iterations=2, random_state=42)
    adata.obs[f'leiden_harmony_{res}'] = adata.obs['leiden'].copy()


In [ ]:
# Cell Identitites were assigned using the Marker genes from the orginal pulbication

# Definations of LOC Genes are as follows: 

# B Cells
# LOC110537828 B-cell antigen receptor complex-associated protein alpha chain [ Oncorhynchus mykiss (rainbow trout) ]
# LOC110510079 B-cell antigen receptor complex-associated protein alpha chain [ Oncorhynchus mykiss (rainbow trout) ]
# LOC110487482 Ig lambda chain C region [ Oncorhynchus mykiss (rainbow trout) ]
# LOC110485215 uncharacterized LOC110485215 [ Oncorhynchus mykiss (rainbow trout) ] IgM heavy chain membrane bound form

# Dendritic Cell (DC) markers and Hematopoietic Progenitor Cell Markers were combined into 
# a single cluster called "Dendritic Cell Progenitors".

# Dendritic Progenitor Cell
# LOC110536334 zinc finger protein 366 [ Oncorhynchus mykiss (rainbow trout) ]
# LOC110512274 MHC class II transactivator [ Oncorhynchus mykiss (rainbow trout) ]
# LOC110486829 CD83 antigen [ Oncorhynchus mykiss (rainbow trout) ]
# LOC110511347 MHC class II transactivator-like [ Oncorhynchus mykiss (rainbow trout) ]
# LOC110535338 fms related receptor tyrosine kinase 3 [ Oncorhynchus mykiss (rainbow trout) ]

# Non-specific cytotoxic cells
# LOC110531658 perforin-1 [ Oncorhynchus mykiss (rainbow trout) ]

# Neutrophils
# LOC110509901 eosinophil peroxidase [ Oncorhynchus mykiss (rainbow trout)

marker_genes_trout = {
    "B cells": ["LOC110537828", "LOC110510079", "cd79b", "LOC110487482", "ccl4", "LOC110485215", "irf4l"],
    "T cells": ["cd3z", "cd4-1", "cd8a", "cd3e", "cd28"],
    "Monocytes": ["cd209", "lyz2", "cd9a"],
    "Dendritic Progenitor Cells": ["oncmyk-dbb", "LOC110512274", "LOC110486829", "LOC110511347", "flt3", "LOC110535338"],
    "Non-specific cytotoxic cells": ["LOC110531658"],
    "Neutrophils": ["LOC110509901"],
    "Thrombocytes": ["gp1bb", "itga2b"]
}

In [ ]:
# After reviewing expression of markers at different resolutions (0.3, 0.5, 0.6, 0.7, 0.8, 1.0, 1.5), 
# 0.5 resolution was selected for cluster annotation

sc.pl.dotplot(
    adata,
    var_names=marker_genes_trout,
    groupby="leiden_harmony_0.5",
    standard_scale='var',
    show=True,
    colorbar_title='Min-Max Scaled Expression',
    save="_harmony_0.5.png"
)

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="leiden_harmony_0.5",
    method="wilcoxon",
    n_genes=25,
    pts=True,
    key_addded="rank_genes"
)

In [ ]:
result = adata.uns["rank_genes_groups"]
groups= result["names"].dtype.names

marker_dfs = []
for cluster in groups:
    df = sc.get.rank_genes_groups_df(
        adata,
        group=cluster,
        key="rank_genes_groups",
        pval_cutoff=0.001,
        log2fc_min=0.25
    )
    df =df.head(25)
    df["cluster"] = cluster
    marker_dfs.append(df)

all_markers = pd.concat(marker_dfs)
all_markers.to_csv("top25_markers_per_cluster.csv", index=False)

In [ ]:
# Cell Type Annotation were done using pre-defined marker genes in conjuction with with top 25 differential expressed genes per cluster

celltype_map = {
    "0": "T Cells",
    "1": "Monocytes",
    "2": "Neutrophils",
    "3": "Dendritic Progenitor Cells",
    "4": "Non-specific cytotoxic cells",
    "5": "B Cells",
    "6": "B Cells",
    "7": "Non-specific cytotoxic cells",
    "8": "Thrombocytes",
    "9": "B Cells"
}

In [ ]:
adata.obs["cell_type"] = adata.obs["leiden_harmony_0.5"].map(celltype_map)

In [ ]:
adata.obs["cell_type"] = pd.Categorical(
    adata.obs["cell_type"],
    categories=list(dict.fromkeys(celltype_map.values()))
)

In [ ]:
# Visualize cell type annotation
sc.pl.umap(
    adata, 
    color="cell_type", 
    title="Cell Type Annotation",
    save="_cell_type_annotation.png"
)

In [ ]:
# "LOC110498448", "LOC110498643" - 	B-cell lymphoma/leukemia 10-like
# "LOC110506216" - TNF receptor-associated factor 6-like

all_genes = [
    "malt1", "malt2", "malt3", "card9", "irak3", "LOC110506216", "iraf1",
    "card9", "card11", "card14", "map3k7a", "map3k7b",
    "LOC110498448", "LOC110498643",
]

genes_in_adata = [g for g in all_genes if g in adata.var_names]
genes_missing = [g for g in all_genes if g not in adata.var_names]

print("Genes found:", genes_in_adata)
print("Genes missing:", genes_missing)

In [ ]:
# 14. Gene Expression Analysis
# Analyzing expression patterns of MALT genes and other immune signaling genes

# Define colors for each gene
gene_colors = {
    'malt1': '#5DADE2',
    'malt2': '#F5B041', 
    'malt3': '#58D68D',
    'card9': '#99A3A4', 
    'irak3': '#E74C3C',
    'LOC110498448': '#F4D03F', 
    'LOC110498643': '#EC7063',
    'LOC110506216': '#BA6B57'
}

In [ ]:
# Order genes
malt_genes = ['malt1', 'malt2', 'malt3']
other_genes = ['card9', 'irak3', 'LOC110498448', 'LOC110498643', 'LOC110506216']
all_genes = malt_genes + other_genes

# Get cell type order from adata
cell_types = list(adata.obs['cell_type'].cat.categories) \
            if hasattr(adata.obs['cell_type'], 'cat') \
            else list(adata.obs['cell_type'].unique())

In [ ]:
sc.pl.matrixplot(adata,
                var_names=all_genes,
                groupby='cell_type',                   
                categories_order=cell_types,  
                standard_scale='var',
                cmap='Blues',
                title='MALT & Immune Signaling Genes - Matrix Plot',
                show=False,
                save=False
                )


fig = plt.gcf()
fig.patch.set_alpha(0.0)
for ax in fig.get_axes():
    ax.patch.set_alpha(0.0)

fig.savefig('figures/_malt_matrixplot.png', transparent=True, bbox_inches='tight')
plt.close(fig)

In [ ]:
sc.pl.dotplot(adata,
                var_names=all_genes,
                groupby='cell_type',                   
                categories_order=cell_types,  
                standard_scale='var',
                title='MALT & Immune Signaling Genes - Matrix Plot',
                show=False,
                save=False)

fig = plt.gcf()
fig.patch.set_alpha(0.0)
for ax in fig.get_axes():
    ax.patch.set_alpha(0.0)

fig.savefig('figures/_malt_dotplot.png', transparent=True, bbox_inches='tight')
plt.close(fig)

In [ ]:
# Save the final annotated AnnData object
adata.write('adata_object/adata_control.h5ad')